# Одна операция — два способа: SQL и pandas

Одна и та же операция разведочного анализа — слева запросом
к базе, справа в pandas. Обе колонки выполняются на одних и тех же данных
кредитного конвейера, и результат сверяется автоматически: памятка не даёт
обещаний, которые нельзя проверить.

SQL считает там, где лежат данные: база читает миллионы строк
и отдаёт десятки. Pandas работает уже с тем, что поместилось в память
тетради. Отсюда практическое правило: **отбор, соединение и агрегация —
запросом; доводка, форма таблицы и график — в pandas.** Ошибка, которая
дорого стоит, — вытащить `SELECT *` на миллион строк и группировать его
в тетради: то же самое база сделает за секунды и вернёт готовый ответ.

Что нужно: поднятый стенд (`./run.sh up`). Данные — те же таблицы кредитного
конвейера, что и в тетради «Конвейер SQL».

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from tools.compare import запрос, сверить, таблица

# Одни и те же данные для обеих колонок памятки.
заявки  = таблица("raw_applications")
этапы   = таблица("raw_stage_events")
решения = таблица("raw_decisions")
выдачи  = таблица("raw_disbursements")

print("заявки:", заявки.shape, "· этапы:", этапы.shape,
      "· решения:", решения.shape, "· выдачи:", выдачи.shape)

заявки: (3672, 9) · этапы: (14203, 6) · решения: (3546, 4) · выдачи: (1821, 3)


Ниже каждая операция идёт одинаково: сначала запрос, потом то же самое в pandas,
потом сверка. Строка «✓ совпадает» под ячейкой означает, что обе колонки дали
один и тот же результат — это проверяется на ваших данных при каждом запуске.

## 1. Размер выгрузки и границы периода

Первый вопрос к незнакомым данным: сколько строк и за какой период. Пока это не проверено, любое число ниже повисает в воздухе.

In [2]:
слева = запрос("""
SELECT count(*)                       AS строк,
       count(DISTINCT application_id) AS заявок,
       min(submitted_at)::date        AS первая,
       max(submitted_at)::date        AS последняя
FROM raw_applications
""")

справа = pd.DataFrame([{
    "строк":  len(заявки),
    "заявок": заявки["application_id"].nunique(),
    "первая": заявки["submitted_at"].min().date(),
    "последняя": заявки["submitted_at"].max().date(),
}])

сверить(слева, справа)

строк,заявок,первая,последняя
3672,3600,2025-01-06,2025-08-03
строк,заявок,первая,последняя
3672,3600,2025-01-06,2025-08-03


✓ совпадает : 1 строк × 4 столбцов


True

> Строк больше, чем заявок, — это уже находка: в выгрузке повторы.

## 2. Какие столбцы и какого они типа

Тип столбца решает, что с ним можно делать. Сумма, сохранённая как текст, искажает среднее.

In [3]:
слева = запрос("""
SELECT column_name AS столбец, data_type AS тип
FROM information_schema.columns
WHERE table_name = 'raw_applications'
ORDER BY ordinal_position
""")

справа = заявки.dtypes

print(слева.to_string(index=False))
print()
print(справа)

         столбец                         тип
  application_id                        text
       client_id                        text
         product                        text
         channel                        text
          region                        text
amount_requested                     numeric
    submitted_at timestamp without time zone
   source_system                        text
         is_test                     boolean

application_id                 str
client_id                      str
product                        str
channel                        str
region                         str
amount_requested           float64
submitted_at        datetime64[us]
source_system                  str
is_test                       bool
dtype: object


> В базе тип задан схемой и гарантирован. В pandas тип угадывается при чтении файла, поэтому `dtypes` смотрят всегда: `object` у колонки с суммами означает, что внутри текст.

## 3. Посмотреть первые строки

Просмотр десятка строк показывает то, чего не видно ни в одной сводке: формат дат, лишние пробелы, служебные значения.

In [4]:
слева = запрос("""
SELECT application_id, product, channel, amount_requested
FROM raw_applications
ORDER BY application_id
LIMIT 5
""")

справа = (заявки.sort_values("application_id")
          [["application_id", "product", "channel", "amount_requested"]]
          .head(5))

сверить(слева, справа)

application_id,product,channel,amount_requested
APP-200000,Автокредит,Корпоративный менеджер,150000
APP-200001,Кредит наличными,Мобильное приложение,800000
APP-200002,Кредит наличными,Партнёрская сеть,800000
APP-200003,Кредитная карта,Отделение,5000000
APP-200004,Кредит наличными,Мобильное приложение,1200000
application_id,product,channel,amount_requested
APP-200000,Автокредит,Корпоративный менеджер,150000.0
APP-200001,Кредит наличными,Мобильное приложение,800000.0
APP-200002,Кредит наличными,Партнёрская сеть,800000.0
APP-200003,Кредитная карта,Отделение,5000000.0


✓ совпадает : 5 строк × 4 столбцов


True

> `LIMIT` без `ORDER BY` возвращает произвольные строки: порядок в таблице базы не определён. То же и с `head()` — он берёт первые строки в текущем порядке кадра.

## 4. Отобрать строки по условию

Любая проверка гипотезы начинается с отбора: один продукт, один канал, один диапазон сумм.

In [5]:
слева = запрос("""
SELECT count(*) AS заявок, round(avg(amount_requested)) AS средняя
FROM raw_applications
WHERE product = 'Автокредит'
  AND amount_requested > 3000000
""")

крупные = заявки[(заявки["product"] == "Автокредит")
                & (заявки["amount_requested"] > 3_000_000)]
справа = pd.DataFrame([{
    "заявок":  len(крупные),
    "средняя": round(крупные["amount_requested"].mean()),
}])

сверить(слева, справа)

заявок,средняя
89,5000000
заявок,средняя
89,5000000


✓ совпадает : 1 строк × 2 столбцов


True

> Скобки вокруг каждого условия в pandas обязательны: `&` выполняется раньше сравнения, и без скобок выражение падает.

## 5. Уникальные значения категории

Пока не видно списка значений, группировка по столбцу бессмысленна: «Отделение» и «отделение » — две разные строки.

In [6]:
слева = запрос("""
SELECT DISTINCT channel AS канал
FROM raw_applications
ORDER BY канал
""")

справа = pd.DataFrame({"канал": sorted(заявки["channel"].unique())})

сверить(слева, справа)

канал
Корпоративный менеджер
Мобильное приложение
Отделение
Партнёрская сеть
канал
Корпоративный менеджер
Мобильное приложение
Отделение
Партнёрская сеть


✓ совпадает : 4 строк × 1 столбцов


True

> Порядок сортировки строк в базе и в Python может отличаться локалью — если сверяете списки, сортируйте обе стороны одинаково.

## 6. Частоты значений: сколько чего

Самая частая операция разведочного анализа. Показывает и структуру данных, и некорректные значения в них.

In [7]:
слева = запрос("""
SELECT channel        AS канал,
       count(*)       AS заявок
FROM raw_applications
GROUP BY channel
ORDER BY заявок DESC
""")

справа = (заявки["channel"].value_counts()
          .rename_axis("канал").reset_index(name="заявок"))

сверить(слева, справа)

канал,заявок
Мобильное приложение,1230
Отделение,1081
Партнёрская сеть,786
Корпоративный менеджер,575
канал,заявок
Мобильное приложение,1230
Отделение,1081
Партнёрская сеть,786
Корпоративный менеджер,575


✓ совпадает : 4 строк × 2 столбцов


True

> `value_counts()` сразу сортирует по убыванию — это и есть `GROUP BY … ORDER BY count(*) DESC` одной командой.

## 7. Пропуски по столбцам

Пропуск — это не всегда ошибка. Незакрытый этап означает «ещё идёт», и такие строки считают отдельно, а не выбрасывают.

In [8]:
слева = запрос("""
SELECT count(*) FILTER (WHERE left_at IS NULL)    AS без_выхода,
       count(*) FILTER (WHERE actor_role IS NULL) AS без_роли,
       count(*)                                   AS всего
FROM raw_stage_events
""")

справа = pd.DataFrame([{
    "без_выхода": int(этапы["left_at"].isna().sum()),
    "без_роли":   int(этапы["actor_role"].isna().sum()),
    "всего":      len(этапы),
}])

сверить(слева, справа)

без_выхода,без_роли,всего
298,0,14203
без_выхода,без_роли,всего
298,0,14203


✓ совпадает : 1 строк × 3 столбцов


True

> `FILTER (WHERE …)` — способ посчитать несколько условий одним проходом по таблице. В pandas тому же соответствует `isna().sum()` по каждому столбцу.

## 8. Сколько строк лишние: повторы

Повторы приезжают при склейке выгрузок из разных систем и завышают всё, что считается по строкам.

In [9]:
слева = запрос("""
SELECT count(*)                                  AS строк,
       count(DISTINCT application_id)            AS заявок,
       count(*) - count(DISTINCT application_id) AS лишних
FROM raw_applications
""")

справа = pd.DataFrame([{
    "строк":  len(заявки),
    "заявок": заявки["application_id"].nunique(),
    "лишних": int(заявки["application_id"].duplicated().sum()),
}])

сверить(слева, справа)

строк,заявок,лишних
3672,3600,72
строк,заявок,лишних
3672,3600,72


✓ совпадает : 1 строк × 3 столбцов


True

> `duplicated()` по умолчанию помечает все повторы, кроме первого, — поэтому его сумма и есть число лишних строк.

## 9. Убрать повторы, оставив первую запись

Правило «какую из копий оставляем» задаёт аналитик, а не инструмент: обычно самую раннюю или самую свежую по времени.

In [10]:
слева = запрос("""
SELECT count(*) AS осталось
FROM (
    SELECT DISTINCT ON (application_id) application_id
    FROM raw_applications
    ORDER BY application_id, submitted_at
) t
""")

без_повторов = (заявки.sort_values(["application_id", "submitted_at"])
                 .drop_duplicates("application_id", keep="first"))
справа = pd.DataFrame([{"осталось": len(без_повторов)}])

сверить(слева, справа)

осталось
3600
осталось
3600


✓ совпадает : 1 строк × 1 столбцов


True

> `DISTINCT ON` — приём PostgreSQL: он оставляет первую строку в порядке `ORDER BY`. В других диалектах то же делают через `ROW_NUMBER() OVER (PARTITION BY …)` и отбор `= 1`.

## 10. Привести тип: суммы к числу

В выгрузке из файла суммы почти всегда приходят текстом — с пробелами и запятой. Пока не приведены, среднее считать нельзя.

In [11]:
слева = запрос("""
SELECT round(sum(amount_requested)) AS сумма
FROM raw_applications
""")

суммы = pd.to_numeric(заявки["amount_requested"], errors="coerce")
справа = pd.DataFrame([{"сумма": round(суммы.sum())}])

сверить(слева, справа)

сумма
5595850000
сумма
5595850000


✓ совпадает : 1 строк × 1 столбцов


True

> `errors="coerce"` превращает неразобранное в `NaN` вместо ошибки — и число таких `NaN` сразу показывает, сколько значений не приводились к типу. В базе тип задан схемой, приводить нечего.

## 11. Группировка с несколькими метриками

Основной рабочий инструмент: разрез плюс два-три показателя.

In [12]:
слева = запрос("""
SELECT channel                     AS канал,
       count(*)                    AS заявок,
       round(avg(amount_requested)) AS средняя,
       round(sum(amount_requested)) AS сумма
FROM raw_applications
GROUP BY channel
ORDER BY заявок DESC
""")

справа = (заявки.groupby("channel")
          .agg(заявок=("application_id", "count"),
               средняя=("amount_requested", "mean"),
               сумма=("amount_requested", "sum"))
          .round(0).reset_index()
          .rename(columns={"channel": "канал"})
          .sort_values("заявок", ascending=False)
          .reset_index(drop=True))

сверить(слева, справа)

канал,заявок,средняя,сумма
Мобильное приложение,1230,1511138,1858700000
Отделение,1081,1547549,1672900000
Партнёрская сеть,786,1530852,1203250000
Корпоративный менеджер,575,1497391,861000000
канал,заявок,средняя,сумма
Мобильное приложение,1230,1511138.0,1858700000.0
Отделение,1081,1547549.0,1672900000.0
Партнёрская сеть,786,1530852.0,1203250000.0
Корпоративный менеджер,575,1497391.0,861000000.0


✓ совпадает : 4 строк × 4 столбцов


True

> Именованная агрегация в pandas (`заявок=(столбец, функция)`) — прямой аналог `AS` в SQL: столбцы сразу называются по-человечески.

## 12. Фильтр после агрегации

«Показать только те группы, где заявок больше тысячи» — это условие на результат, а не на строки.

In [13]:
слева = запрос("""
SELECT channel   AS канал,
       count(*)  AS заявок
FROM raw_applications
GROUP BY channel
HAVING count(*) > 1000
ORDER BY заявок DESC
""")

справа = (заявки["channel"].value_counts()
          .rename_axis("канал").reset_index(name="заявок")
          .query("заявок > 1000")
          .reset_index(drop=True))

сверить(слева, справа)

канал,заявок
Мобильное приложение,1230
Отделение,1081
канал,заявок
Мобильное приложение,1230
Отделение,1081


✓ совпадает : 2 строк × 2 столбцов


True

> `WHERE` отбирает строки до группировки, `HAVING` — группы после. В pandas это просто фильтр по уже посчитанному кадру.

## 13. Сортировка и первые N

Рейтинг — самая частая просьба руководителя: «покажи топ-5».

In [14]:
слева = запрос("""
SELECT region                       AS регион,
       round(sum(amount_requested)) AS сумма
FROM raw_applications
GROUP BY region
ORDER BY сумма DESC
LIMIT 5
""")

справа = (заявки.groupby("region")["amount_requested"].sum()
          .round(0).rename_axis("регион").reset_index(name="сумма")
          .sort_values("сумма", ascending=False)
          .head(5).reset_index(drop=True))

сверить(слева, справа)

регион,сумма
Центральный,1698700000
Приволжский,1199950000
Северо-Западный,1006800000
Южный,879800000
Сибирский,810600000
регион,сумма
Центральный,1698700000.0
Приволжский,1199950000.0
Северо-Западный,1006800000.0
Южный,879800000.0


✓ совпадает : 5 строк × 2 столбцов


True

> При равных значениях на границе топа порядок не определён ни там, ни там: если это важно, добавляйте второй ключ сортировки.

## 14. Соединить две таблицы

Данные почти никогда не лежат в одной таблице: заявки в одной, решения по ним в другой.

In [15]:
слева = запрос("""
SELECT a.channel   AS канал,
       count(*)    AS отказов
FROM raw_applications a
JOIN raw_decisions d ON d.application_id = a.application_id
WHERE d.decision = 'Отказ'
GROUP BY a.channel
ORDER BY отказов DESC
""")

вместе = заявки.merge(решения, on="application_id", how="inner")
справа = (вместе[вместе["decision"] == "Отказ"]["channel"]
          .value_counts().rename_axis("канал").reset_index(name="отказов"))

сверить(слева, справа)

канал,отказов
Мобильное приложение,558
Отделение,537
Партнёрская сеть,366
Корпоративный менеджер,267
канал,отказов
Мобильное приложение,558
Отделение,537
Партнёрская сеть,366
Корпоративный менеджер,267


✓ совпадает : 4 строк × 2 столбцов


True

> `how="inner"` — это `JOIN`, `how="left"` — `LEFT JOIN`. После соединения всегда проверяйте число строк: если оно выросло, ключ не уникален и результат уже задвоен.

## 15. Разбивка по месяцам

Динамика отвечает на вопрос «стало лучше или хуже», а одно число за период — нет.

In [16]:
слева = запрос("""
SELECT to_char(date_trunc('month', submitted_at), 'YYYY-MM') AS месяц,
       count(*)                                          AS заявок
FROM raw_applications
GROUP BY 1
ORDER BY 1
""")

справа = (заявки["submitted_at"].dt.to_period("M").astype(str)
          .value_counts().sort_index()
          .rename_axis("месяц").reset_index(name="заявок"))

сверить(слева, справа)

месяц,заявок
2025-01,468
2025-02,487
2025-03,522
2025-04,540
2025-05,553
2025-06,526
2025-07,529
2025-08,47
месяц,заявок
2025-01,468


✓ совпадает : 8 строк × 2 столбцов


True

> `date_trunc` округляет дату вниз до начала периода. В pandas ту же роль играет `to_period("M")`, а `dt.month` — не то же самое: он склеит январь двух разных лет.

## 16. Сводная таблица: два разреза сразу

Пересечение двух признаков показывает то, чего не видно по каждому в отдельности: например, что доля канала неодинакова у разных продуктов.

In [17]:
слева = запрос("""
SELECT product AS продукт,
       count(*) FILTER (WHERE channel = 'Отделение')            AS отделение,
       count(*) FILTER (WHERE channel = 'Мобильное приложение') AS приложение
FROM raw_applications
GROUP BY product
ORDER BY продукт
""")

сводная = заявки.pivot_table(index="product", columns="channel",
                            values="application_id", aggfunc="count",
                            fill_value=0)
справа = (сводная[["Отделение", "Мобильное приложение"]]
          .rename(columns={"Отделение": "отделение",
                           "Мобильное приложение": "приложение"})
          .rename_axis("продукт").reset_index()
          .sort_values("продукт").reset_index(drop=True))

сверить(слева, справа)

✓ совпадает : 4 строк × 3 столбцов


True

> `pivot_table` — самый быстрый способ увидеть пересечение признаков. В SQL то же делают через `FILTER (WHERE …)` или `CASE WHEN` внутри агрегата.

## 17. Доля от общего

Абсолютные числа сравнивать нельзя, если группы разного размера. Доля — минимальная нормировка.

In [18]:
слева = запрос("""
SELECT channel                                          AS канал,
       count(*)                                         AS заявок,
       round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS доля
FROM raw_applications
GROUP BY channel
ORDER BY заявок DESC
""")

по_каналу = (заявки["channel"].value_counts()
             .rename_axis("канал").reset_index(name="заявок"))
по_каналу["доля"] = (100 * по_каналу["заявок"]
                     / по_каналу["заявок"].sum()).round(1)
справа = по_каналу

сверить(слева, справа)

канал,заявок,доля
Мобильное приложение,1230,33.5
Отделение,1081,29.4
Партнёрская сеть,786,21.4
Корпоративный менеджер,575,15.7
канал,заявок,доля
Мобильное приложение,1230,33.5
Отделение,1081,29.4
Партнёрская сеть,786,21.4
Корпоративный менеджер,575,15.7


✓ совпадает : 4 строк × 3 столбцов


True

> `sum(count(*)) OVER ()` — оконная функция: считает итог по всем группам, не схлопывая их. В pandas это просто деление на сумму столбца.

## 18. Медиана и перцентиль

Среднее по длительностям обманывает: один долгий случай тянет его вверх. Медиана — типичный срок, P90 — тот, в который укладывается почти всё.

In [19]:
слева = запрос("""
SELECT stage AS этап,
       round(percentile_cont(0.5) WITHIN GROUP (
             ORDER BY EXTRACT(EPOCH FROM (left_at - entered_at)) / 3600)::numeric, 1) AS медиана,
       round(percentile_cont(0.9) WITHIN GROUP (
             ORDER BY EXTRACT(EPOCH FROM (left_at - entered_at)) / 3600)::numeric, 1) AS p90
FROM raw_stage_events
WHERE left_at IS NOT NULL AND left_at > entered_at
GROUP BY stage
ORDER BY медиана DESC
""")

закрытые = этапы[этапы["left_at"].notna()
                  & (этапы["left_at"] > этапы["entered_at"])].copy()
закрытые["часы"] = ((закрытые["left_at"] - закрытые["entered_at"])
                    .dt.total_seconds() / 3600)
справа = (закрытые.groupby("stage")["часы"]
          .agg(медиана=lambda s: s.quantile(0.5),
               p90=lambda s: s.quantile(0.9))
          .round(1).rename_axis("этап").reset_index()
          .sort_values("медиана", ascending=False)
          .reset_index(drop=True))

сверить(слева, справа)

этап,медиана,p90
Андеррайтинг,40.6,78.1
Подготовка документов,13.0,23.7
Выдача,8.1,14.0
Скоринг,2.8,5.1
Приём заявки,2.0,3.8
этап,медиана,p90
Андеррайтинг,40.6,78.1
Подготовка документов,13.0,23.7
Выдача,8.1,14.0
Скоринг,2.8,5.1


✓ совпадает : 5 строк × 3 столбцов


True

> `percentile_cont` в PostgreSQL и `quantile()` в pandas интерполируют одинаково — линейно. `percentile_disc` вернул бы существующее значение из выборки, и числа разошлись бы.

## 19. Условная колонка: разложить по корзинам

Непрерывную величину почти всегда режут на группы: малые, средние и крупные заявки ведут себя по-разному.

In [20]:
слева = запрос("""
SELECT CASE WHEN amount_requested < 500000  THEN '1. до 500 тыс'
            WHEN amount_requested < 2000000 THEN '2. 0,5–2 млн'
            ELSE                                 '3. свыше 2 млн'
       END      AS размер,
       count(*) AS заявок
FROM raw_applications
GROUP BY размер
ORDER BY размер
""")

размер = pd.cut(заявки["amount_requested"],
                 bins=[-1, 499_999, 1_999_999, float("inf")],
                 labels=["1. до 500 тыс", "2. 0,5–2 млн", "3. свыше 2 млн"])
справа = (размер.value_counts().sort_index()
          .rename_axis("размер").reset_index(name="заявок"))

сверить(слева, справа)

размер,заявок
1. до 500 тыс,997
"2. 0,5–2 млн",1602
3. свыше 2 млн,1073
размер,заявок
1. до 500 тыс,997
"2. 0,5–2 млн",1602
3. свыше 2 млн,1073


✓ совпадает : 3 строк × 2 столбцов


True

> Границы корзин задаёт аналитик, и они всегда спорные — поэтому их проговаривают вслух вместе с выводом, а не прячут в код.

## 20. Оконная функция: первая заявка клиента

«Первый», «последний», «предыдущий» — это всегда окно: строки нумеруются внутри группы, а не по всей таблице.

In [21]:
слева = запрос("""
SELECT count(*) AS первых_заявок
FROM (
    SELECT ROW_NUMBER() OVER (PARTITION BY client_id
                              ORDER BY submitted_at) AS n
    FROM raw_applications
) t
WHERE n = 1
""")

номер = (заявки.sort_values("submitted_at")
         .groupby("client_id").cumcount() + 1)
справа = pd.DataFrame([{"первых_заявок": int((номер == 1).sum())}])

сверить(слева, справа)

первых_заявок
3439
первых_заявок
3439


✓ совпадает : 1 строк × 1 столбцов


True

> Оконная функция не схлопывает строки: рядом с каждой заявкой появляется её номер, и дальше можно отобрать любой. В pandas тот же приём — `cumcount()` после сортировки.

## 21. Выбросы по правилу полутора размахов

Классический способ отделить «долго» от «невозможно долго». Найденное — повод для вопроса, а не для удаления строки.

In [22]:
слева = запрос("""
WITH ч AS (
    SELECT EXTRACT(EPOCH FROM (left_at - entered_at)) / 3600 AS часы
    FROM raw_stage_events
    WHERE stage = 'Андеррайтинг' AND left_at > entered_at
), г AS (
    SELECT percentile_cont(0.25) WITHIN GROUP (ORDER BY часы) AS q1,
           percentile_cont(0.75) WITHIN GROUP (ORDER BY часы) AS q3
    FROM ч
)
SELECT count(*) AS выбросов,
       round((SELECT (q3 + 1.5 * (q3 - q1))::numeric FROM г), 1) AS порог
FROM ч, г
WHERE ч.часы > г.q3 + 1.5 * (г.q3 - г.q1)
""")

андеррайтинг = этапы[(этапы["stage"] == "Андеррайтинг")
                     & (этапы["left_at"] > этапы["entered_at"])]
часы = ((андеррайтинг["left_at"] - андеррайтинг["entered_at"])
        .dt.total_seconds() / 3600)
q1, q3 = часы.quantile(0.25), часы.quantile(0.75)
порог = q3 + 1.5 * (q3 - q1)
справа = pd.DataFrame([{"выбросов": int((часы > порог).sum()),
                        "порог": round(порог, 1)}])

сверить(слева, справа)

выбросов,порог
139,94.7
выбросов,порог
139,94.7


✓ совпадает : 1 строк × 2 столбцов


True

> Правило работает на распределениях без длинного хвоста. Там, где хвост есть по природе процесса, оно пометит выбросами нормальные случаи — тогда границу задают перцентилем, например P99.

## Что из этого следует

Обе колонки дают одинаковые числа, поэтому выбор между ними — не вопрос
правильности, а вопрос места и объёма:

- **Данных много, ответ маленький** — считает база. Отбор, соединение,
  агрегация по миллионам строк живут в запросе.
- **Данных уже мало, нужна форма** — работает pandas. Доводка таблицы,
  расчёт производных колонок, подготовка к графику.
- **Правило чистки, которым пользуются все** — представление в базе, а не
  ячейка в чужой тетради: так оно живёт в одном месте и не расходится
  между отчётами.

## Задание

Возьмите любую операцию из памятки и повторите её на второй паре таблиц —
портфеле инициатив (`initiative_passport`, `initiative_fact`). Сверка покажет,
получилось ли: если строка «✓ совпадает» не появилась, расходится либо
условие отбора, либо тип столбца.